# PJM Forecast Summary

Regional load, solar, and wind forecast tables for a given target date.

- **Load**: Meteologica zonal breakdown + PJM (ISO) vs Meteologica RTO comparison
- **Solar**: PJM (grid-scale + BTM) vs Meteologica
- **Wind**: PJM vs Meteologica

## 1. Setup & Configuration

In [1]:
import sys
import logging
from pathlib import Path

import pandas as pd
import numpy as np

# ── Repo root & backend imports ──
REPO_ROOT = Path.cwd().parents[1]
SQL_DIR = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "backend"))

from src.utils.azure_postgresql import pull_from_db

try:
    from src.utils.logging_utils import init_logging
    pl = init_logging(name="forecast_summary", log_to_file=False, level=logging.INFO)
    logger = pl.logger
except Exception:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
    logger = logging.getLogger("forecast_summary")

# ── Configurable Parameters ──
TARGET_DATE = pd.Timestamp.now().normalize()
FORECAST_DATE = TARGET_DATE + pd.Timedelta(days=1)
MAX_EXECUTION_HOUR = 10

SQL_PARAMS = dict(
    execution_date=TARGET_DATE.date(),
    forecast_date=FORECAST_DATE.date(),
    max_execution_hour=MAX_EXECUTION_HOUR,
)

logger.info(f"Execution date: {TARGET_DATE.date()}")
logger.info(f"Forecast date:  {FORECAST_DATE.date()}")


def read_sql(filename, **extra_params):
    """Read a .sql file and format with params."""
    params = {**SQL_PARAMS, **extra_params}
    return (SQL_DIR / filename).read_text().format(**params)

INFO:root:CONFIG_DIR: c:\Users\AidanKeaveny\Documents\github\helioscta-pjm-da\backend\src
INFO:forecast_summary:Execution date: 2026-03-11
INFO:forecast_summary:Forecast date:  2026-03-12


## 2. Data Pull

In [2]:
# ── Load Forecasts ──
df_met_zonal = pull_from_db(query=read_sql("meteologica_zonal_load.sql"))
df_pjm_load = pull_from_db(query=read_sql("pjm_rto_load.sql"))
df_met_load = pull_from_db(query=read_sql("meteologica_rto_load.sql"))

# ── Solar Forecasts ──
df_pjm_solar = pull_from_db(query=read_sql("pjm_solar.sql"))
df_met_solar = pull_from_db(query=read_sql("meteologica_solar.sql"))

# ── Wind Forecasts ──
df_pjm_wind = pull_from_db(query=read_sql("pjm_wind.sql"))
df_met_wind = pull_from_db(query=read_sql("meteologica_wind.sql"))

for name, df in {
    "Met Zonal Load": df_met_zonal, "PJM RTO Load": df_pjm_load,
    "Met RTO Load": df_met_load, "PJM Solar": df_pjm_solar,
    "Met Solar": df_met_solar, "PJM Wind": df_pjm_wind,
    "Met Wind": df_met_wind,
}.items():
    rows = len(df) if df is not None else 0
    (logger.info if rows > 0 else logger.warning)(f"{name}: {rows} rows")

INFO:forecast_summary:Met Zonal Load: 432 rows
INFO:forecast_summary:PJM RTO Load: 24 rows
INFO:forecast_summary:Met RTO Load: 24 rows
INFO:forecast_summary:PJM Solar: 24 rows
INFO:forecast_summary:Met Solar: 24 rows
INFO:forecast_summary:PJM Wind: 24 rows
INFO:forecast_summary:Met Wind: 24 rows


## 3. Helpers

In [3]:
# ── Zone mapping: Meteologica region -> display name ──
ZONE_MAP = {
    'MIDATL_AE': 'AE', 'WEST_AEP': 'AEP', 'WEST_DAY': 'AEP',
    'WEST_AP': 'APS', 'WEST_ATSI': 'ATSI', 'MIDATL_BC': 'BGE',
    'WEST_CE': 'COMED', 'WEST_DEOK': 'DEOK', 'MIDATL_DPL': 'DPL',
    'SOUTH_DOM': 'DOM', 'WEST_DUQ': 'DUQ', 'MIDATL_JC': 'JCPL',
    'MIDATL_ME': 'METED', 'MIDATL_PE': 'PECO', 'MIDATL_PN': 'PENELEC',
    'MIDATL_PEP': 'PEPCO', 'MIDATL_PL': 'PPL', 'MIDATL_PS': 'PSEG',
}

ZONE_ORDER = [
    'AE', 'AEP', 'APS', 'ATSI', 'BGE', 'COMED', 'DEOK', 'DPL',
    'DOM', 'DUQ', 'JCPL', 'METED', 'PECO', 'PENELEC', 'PEPCO', 'PPL', 'PSEG',
]


def fmt_mw(v):
    """Format MW values: integers with commas, floats with 1 decimal."""
    if isinstance(v, str) or v == '':
        return v
    if pd.isna(v):
        return ''
    if isinstance(v, float) and round(v, 1) != round(v, 0):
        return f'{v:,.1f}'
    return f'{int(v):,}'


def color_diff(val):
    """Color DIFF values: green positive, red negative."""
    try:
        v = float(val) if not isinstance(val, (int, float)) else val
    except (ValueError, TypeError):
        return ''
    if pd.isna(v):
        return ''
    if v > 0:
        return 'background-color: #2d5a2d; color: #90EE90'
    elif v < 0:
        return 'background-color: #5a2d2d; color: #FFB6C1'
    return ''


TABLE_STYLES = [
    {'selector': 'th',
     'props': 'background-color: #1a365d; color: white; font-weight: bold; '
              'text-align: center; padding: 6px 10px; border: 1px solid #2d4a7c; '
              'font-size: 12px; white-space: nowrap'},
    {'selector': 'td',
     'props': 'text-align: right; padding: 4px 8px; border: 1px solid #2d4a7c; '
              'font-size: 11px; background-color: #0f1923; color: #e0e0e0; '
              'white-space: nowrap'},
    {'selector': 'caption',
     'props': 'font-size: 16px; font-weight: bold; text-align: left; '
              'padding: 10px 0; color: #FFD700'},
    {'selector': 'tr:last-child td',
     'props': 'background-color: #1a365d; color: #FFD700; font-weight: bold; '
              'border-top: 2px solid #FFD700'},
]

## 4. Load Forecast (MW) — Regional Breakdown

In [4]:
# ── Build zonal pivot ──
df_z = df_met_zonal.copy()
df_z['zone'] = df_z['region'].map(ZONE_MAP)
df_z['forecast_load_mw'] = pd.to_numeric(df_z['forecast_load_mw'], errors='coerce')
df_z = df_z.groupby(['hour_ending', 'zone'])['forecast_load_mw'].sum().reset_index()

load = df_z.pivot(index='hour_ending', columns='zone', values='forecast_load_mw')
load = load.reindex(columns=ZONE_ORDER).fillna(0)
load.index = load.index.astype(int)
load = load.sort_index()

# ── Comparison columns ──
load['PJM TOTAL'] = load[ZONE_ORDER].sum(axis=1)
load['PJM'] = pd.to_numeric(
    df_pjm_load.set_index('hour_ending')['forecast_load_mw']
).reindex(load.index)
load['Meteologica'] = pd.to_numeric(
    df_met_load.set_index('hour_ending')['forecast_load_mw']
).reindex(load.index)
load['DIFF'] = load['PJM'] - load['Meteologica']

mw_cols = ZONE_ORDER + ['PJM TOTAL', 'PJM', 'Meteologica', 'DIFF']

# ── Build display table ──
rows = []
for he in load.index:
    r = {'Hr': he - 1, 'Time': f'{he - 1:02d}:00'}
    for col in mw_cols:
        r[col] = int(round(load.loc[he, col]))
    rows.append(r)

# Daily totals (GWh)
totals = {'Hr': '', 'Time': 'DAILY GWh'}
for col in mw_cols:
    totals[col] = round(load[col].sum() / 1000, 1)
rows.append(totals)

df_load = pd.DataFrame(rows)

# ── Style & display ──
styled = (df_load.style
    .format(fmt_mw, subset=mw_cols)
    .map(color_diff, subset=['DIFF'])
    .set_table_styles(TABLE_STYLES)
    .set_caption(f'LOAD FORECAST (MW) — {FORECAST_DATE.date()}')
    .hide(axis='index')
)
display(styled)

Hr,Time,AE,AEP,APS,ATSI,BGE,COMED,DEOK,DPL,DOM,DUQ,JCPL,METED,PECO,PENELEC,PEPCO,PPL,PSEG,PJM TOTAL,PJM,Meteologica,DIFF
0,00:00,907,"16,293","4,521","6,734","2,679","10,071","2,382","1,486","13,970","1,159","1,957","1,414","3,322","1,578","2,556","3,641","4,131","78,801","78,523","80,134","-1,611"
1,01:00,875,"16,053","4,422","6,656","2,546","9,757","2,339","1,430","13,442","1,142","1,855","1,374","3,207","1,550","2,407","3,540","3,953","76,548","76,532","77,882","-1,350"
2,02:00,856,"16,015","4,383","6,669","2,472","9,568","2,342","1,401","13,098","1,143","1,805","1,345","3,108","1,549","2,330","3,451","3,872","75,407","75,307","76,777","-1,470"
3,03:00,853,"16,076","4,421","6,694","2,426","9,470","2,357","1,380","12,937","1,156","1,766","1,328","3,083","1,565","2,269","3,428","3,793","75,002","74,932","76,436","-1,504"
4,04:00,856,"16,400","4,576","6,846","2,428","9,499","2,412","1,406","12,933","1,187","1,779","1,355","3,123","1,628","2,278","3,503","3,814","76,023","76,394","77,528","-1,134"
5,05:00,899,"17,240","4,874","7,214","2,559","9,678","2,575","1,486","13,228","1,249","1,886","1,452","3,310","1,763","2,414","3,763","3,957","79,547","80,223","81,189",-966
6,06:00,972,"18,511","5,309","7,745","2,817","10,209","2,817","1,652","13,902","1,335","2,085","1,610","3,607","1,964","2,664","4,240","4,248","85,687","86,766","87,519",-753
7,07:00,"1,008","19,353","5,579","8,201","2,947","11,042","3,004","1,711","14,245","1,407","2,205","1,695","3,839","2,084","2,811","4,527","4,458","90,116","91,362","92,042",-680
8,08:00,"1,002","19,666","5,766","8,325","3,062","11,493","3,083","1,750","14,429","1,457","2,276","1,776","3,965","2,128","2,953","4,745","4,574","92,450","93,197","94,418","-1,221"
9,09:00,"1,006","19,785","5,907","8,400","3,180","11,293","3,088","1,796","14,587","1,494","2,347","1,834","4,094","2,161","3,045","4,932","4,724","93,673","93,261","95,678","-2,417"


## 5. Solar Forecast (MW)

In [5]:
# ── Build solar comparison ──
solar = df_pjm_solar[['hour_ending', 'solar_forecast', 'solar_forecast_btm']].copy()
solar.columns = ['hour_ending', 'PJM Grid-Scale', 'PJM BTM']
for c in ['PJM Grid-Scale', 'PJM BTM']:
    solar[c] = pd.to_numeric(solar[c], errors='coerce').fillna(0)
solar['PJM Total'] = solar['PJM Grid-Scale'] + solar['PJM BTM']
solar = solar.set_index('hour_ending').sort_index()
solar.index = solar.index.astype(int)

met_sol = pd.to_numeric(
    df_met_solar.set_index('hour_ending')['forecast_generation_mw']
)
met_sol.index = met_sol.index.astype(int)
solar['Meteologica'] = met_sol.reindex(solar.index).fillna(0)
solar['DIFF'] = solar['PJM Total'] - solar['Meteologica']

sol_cols = ['PJM Grid-Scale', 'PJM BTM', 'PJM Total', 'Meteologica', 'DIFF']

# ── Build display table ──
rows = []
for he in solar.index:
    r = {'Hr': he - 1, 'Time': f'{he - 1:02d}:00'}
    for col in sol_cols:
        r[col] = int(round(solar.loc[he, col]))
    rows.append(r)

totals = {'Hr': '', 'Time': 'DAILY GWh'}
for col in sol_cols:
    totals[col] = round(solar[col].sum() / 1000, 1)
rows.append(totals)

df_solar = pd.DataFrame(rows)

styled = (df_solar.style
    .format(fmt_mw, subset=sol_cols)
    .map(color_diff, subset=['DIFF'])
    .set_table_styles(TABLE_STYLES)
    .set_caption(f'SOLAR FORECAST (MW) — {FORECAST_DATE.date()}')
    .hide(axis='index')
)
display(styled)

Hr,Time,PJM Grid-Scale,PJM BTM,PJM Total,Meteologica,DIFF
0,00:00,0,0,0,0,0
1,01:00,0,0,0,0,0
2,02:00,0,0,0,0,0
3,03:00,0,0,0,0,0
4,04:00,0,0,0,0,0
5,05:00,0,0,0,0,0
6,06:00,0,0,0,0,0
7,07:00,24,42,67,64,3
8,08:00,"1,134",365,"1,499","1,055",444
9,09:00,"4,777",799,"5,576","3,524","2,052"


## 6. Wind Forecast (MW)

In [6]:
# ── Build wind comparison ──
wind = df_pjm_wind[['hour_ending', 'wind_forecast']].copy()
wind.columns = ['hour_ending', 'PJM']
wind['PJM'] = pd.to_numeric(wind['PJM'], errors='coerce').fillna(0)
wind = wind.set_index('hour_ending').sort_index()
wind.index = wind.index.astype(int)

met_wnd = pd.to_numeric(
    df_met_wind.set_index('hour_ending')['forecast_generation_mw']
)
met_wnd.index = met_wnd.index.astype(int)
wind['Meteologica'] = met_wnd.reindex(wind.index).fillna(0)
wind['DIFF'] = wind['PJM'] - wind['Meteologica']

wnd_cols = ['PJM', 'Meteologica', 'DIFF']

# ── Build display table ──
rows = []
for he in wind.index:
    r = {'Hr': he - 1, 'Time': f'{he - 1:02d}:00'}
    for col in wnd_cols:
        r[col] = int(round(wind.loc[he, col]))
    rows.append(r)

totals = {'Hr': '', 'Time': 'DAILY GWh'}
for col in wnd_cols:
    totals[col] = round(wind[col].sum() / 1000, 1)
rows.append(totals)

df_wind = pd.DataFrame(rows)

styled = (df_wind.style
    .format(fmt_mw, subset=wnd_cols)
    .map(color_diff, subset=['DIFF'])
    .set_table_styles(TABLE_STYLES)
    .set_caption(f'WIND FORECAST (MW) — {FORECAST_DATE.date()}')
    .hide(axis='index')
)
display(styled)

Hr,Time,PJM,Meteologica,DIFF
0,00:00,"7,130","7,793",-663
1,01:00,"6,685","7,298",-613
2,02:00,"6,229","6,695",-466
3,03:00,"5,787","6,040",-253
4,04:00,"5,139","5,521",-382
5,05:00,"4,562","5,157",-595
6,06:00,"4,315","4,841",-526
7,07:00,"4,205","4,626",-421
8,08:00,"3,793","4,424",-631
9,09:00,"3,248","4,047",-799
